In [1]:
# Cell 1
import httpx
import json
import asyncio
import time
import numpy as np

BASE = "http://localhost:8000"

async def test_all_endpoints():
    async with httpx.AsyncClient() as client:

        print("FASTAPI ENDPOINT TESTS")
        print("=" * 45)

        # Root
        r = await client.get(f"{BASE}/")
        print(f"\n✅ GET / → {r.status_code}")
        print(f"   {r.json()}")

        # Health
        r = await client.get(
            f"{BASE}/health")
        print(f"\n✅ GET /health → "
              f"{r.status_code}")
        print(f"   status: "
              f"{r.json()['status']}")
        print(f"   model : "
              f"{r.json()['model']}")

        # Recommend
        r = await client.post(
            f"{BASE}/recommend",
            json={"user_id": 481,
                  "top_k": 5})
        print(f"\n✅ POST /recommend → "
              f"{r.status_code}")
        data = r.json()
        print(f"   request_id : "
              f"{data['request_id']}")
        print(f"   n_recs     : "
              f"{data['n_recs']}")
        print(f"   latency_ms : "
              f"{data['latency_ms']}")
        for rec in data[
                'recommendations'][:3]:
            print(f"   {rec['rank']}. "
                  f"{rec['title'][:35]} "
                  f"score={rec['score']}")

        # Feedback
        r = await client.post(
            f"{BASE}/feedback",
            json={
                "user_id":  1,
                "movie_id": 356,
                "rating":   4.5,
                "action":   "watch",
            })
        print(f"\n✅ POST /feedback → "
              f"{r.status_code}")
        print(f"   {r.json()['status']}")

        # Metrics
        r = await client.get(
            f"{BASE}/metrics")
        print(f"\n✅ GET /metrics → "
              f"{r.status_code}")
        m = r.json()
        print(f"   total   : "
              f"{m['total_requests']}")
        print(f"   success : "
              f"{m['successful']}")
        print(f"   p50 ms  : "
              f"{m['latency_p50_ms']}")

        return data

result = await test_all_endpoints()

FASTAPI ENDPOINT TESTS

✅ GET / → 200
   {'message': 'Production RecSys API', 'docs': '/docs', 'health': '/health', 'version': '1.0.0'}

✅ GET /health → 200
   status: healthy
   model : HSTU via BentoML (healthy)

✅ POST /recommend → 200
   request_id : 281cdd30
   n_recs     : 5
   latency_ms : 87.47
   1. Heat score=0.0373
   2. Father of the Bride Part II score=0.0521
   3. Waiting to Exhale score=0.8218

✅ POST /feedback → 200
   logged

✅ GET /metrics → 200
   total   : 3
   success : 3
   p50 ms  : 76.6


In [4]:
# Cell 2 — Latency benchmark
async def latency_benchmark(n=20):
    latencies = []
    async with httpx.AsyncClient() as client:
        for i in range(n):
            start = time.time()
            r = await client.post(
                f"{BASE}/recommend",
                json={"user_id": 481,
                      "top_k": 10})
            latencies.append(
                (time.time()-start)*1000)

    print(f"LATENCY BENCHMARK (n={n})")
    print(f"  p50 : "
          f"{np.percentile(latencies,50):.1f}ms")
    print(f"  p95 : "
          f"{np.percentile(latencies,95):.1f}ms")
    print(f"  p99 : "
          f"{np.percentile(latencies,99):.1f}ms")
    print(f"  mean: "
          f"{np.mean(latencies):.1f}ms")

    # Check SLA
    p99 = np.percentile(latencies, 99)
    print(f"\nSLA p99 < 200ms: "
          f"{'✅' if p99 < 200 else '⚠️'}")

    return latencies

lats = await latency_benchmark(20)

LATENCY BENCHMARK (n=20)
  p50 : 45.8ms
  p95 : 81.4ms
  p99 : 130.6ms
  mean: 53.4ms

SLA p99 < 200ms: ✅


In [5]:
# Cell 3 — Check OpenAPI docs
import httpx

async with httpx.AsyncClient() as client:
    r = await client.get(
        "http://localhost:8000/openapi.json")
    schema = r.json()

print("OpenAPI Schema:")
print(f"  Title   : {schema['info']['title']}")
print(f"  Version : {schema['info']['version']}")
print(f"  Endpoints:")
for path in schema['paths']:
    methods = list(
        schema['paths'][path].keys())
    print(f"    {path} → {methods}")

OpenAPI Schema:
  Title   : Production RecSys API
  Version : 1.0.0
  Endpoints:
    /health → ['get']
    /readiness → ['get']
    /recommend → ['post']
    /feedback → ['post']
    /metrics → ['get']
    / → ['get']


In [6]:
# Cell 4 — Save results
import json
import numpy as np

day30_results = {
    "service":    "FastAPI",
    "port":       8000,
    "upstream":   "BentoML:3001",
    "endpoints": [
        "GET  /",
        "GET  /health",
        "GET  /readiness",
        "POST /recommend",
        "POST /feedback",
        "GET  /metrics",
        "GET  /docs",
        "GET  /redoc",
    ],
    "features": [
        "rate_limiting_100_per_min",
        "api_key_auth",
        "cors_middleware",
        "request_id_tracking",
        "structured_json_logging",
        "openapi_docs",
        "pydantic_validation",
    ],
    "latency": {
        "p50_ms": round(float(
            np.percentile(lats, 50)), 1),
        "p95_ms": round(float(
            np.percentile(lats, 95)), 1),
        "p99_ms": round(float(
            np.percentile(lats, 99)), 1),
    },
}

with open(
        '../../data/processed/'
        'day30_results.json', 'w') as f:
    json.dump(day30_results, f, indent=2)

print("✅ Day 30 results saved")
print(json.dumps(day30_results, indent=2))

✅ Day 30 results saved
{
  "service": "FastAPI",
  "port": 8000,
  "upstream": "BentoML:3001",
  "endpoints": [
    "GET  /",
    "GET  /health",
    "GET  /readiness",
    "POST /recommend",
    "POST /feedback",
    "GET  /metrics",
    "GET  /docs",
    "GET  /redoc"
  ],
  "features": [
    "rate_limiting_100_per_min",
    "api_key_auth",
    "cors_middleware",
    "request_id_tracking",
    "structured_json_logging",
    "openapi_docs",
    "pydantic_validation"
  ],
  "latency": {
    "p50_ms": 45.8,
    "p95_ms": 81.4,
    "p99_ms": 130.6
  }
}
